In [ ]:
# Update packages
!apt-get update -qq

# Install required dependency
!apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!pip install -q ollama pandas openpyxl tqdm requests

In [ ]:
import os
import re
import time
import signal
import subprocess

import ollama
import pandas as pd
import requests
from tqdm import tqdm

In [ ]:
MODEL = "model_tag from ollama"

DATASET = "link of the dataset"
OUTPUT_DIR = "/content"

CHECKPOINT_INTERVAL = 25
RESTART_INTERVAL = 40

SYSTEM_PROMPT_FACTUAL = """
You are a helpful AI assistant.

Always answer in the same language as the question.

Provide only the factual answer.

Do not explain.

Do not justify.

Keep the answer concise.
"""

SYSTEM_PROMPT_TF = """
You are a helpful AI assistant.

The question is a True/False statement. Answer only with "True" or "False".

Do not explain.

Do not justify.

Do not add anything else.
"""

SYSTEM_PROMPT_SI = """
You are a helpful AI assistant.

The question presents a statement that may be semantically correct or incorrect. Answer only with "Correct" or "Incorrect".

Do not explain.

Do not justify.

Do not add anything else.
"""

# ---- Prompt type selection ----
print("Select question type:")
print("1. Factual")
print("2. True/False")
print("3. Semantically Incorrect")

choice = input("Enter your choice (1/2/3): ").strip()

if choice == "1":
    SYSTEM_PROMPT = SYSTEM_PROMPT_FACTUAL
    selected_type = "Factual"
elif choice == "2":
    SYSTEM_PROMPT = SYSTEM_PROMPT_TF
    selected_type = "True/False"
elif choice == "3":
    SYSTEM_PROMPT = SYSTEM_PROMPT_SI
    selected_type = "Semantically Incorrect"
else:
    raise ValueError("Invalid choice. Please enter 1, 2, or 3.")

print(f"\n✅ Confirmed: You have selected '{selected_type}' prompt.\n")

In [ ]:
server = None

def _unload_model():
    """Ask Ollama to release the model from memory gracefully before killing anything."""
    try:
        requests.post(
            "http://localhost:11434/api/generate",
            json={"model": MODEL, "keep_alive": 0},
            timeout=10
        )
        time.sleep(2)
    except Exception:
        pass  # server may already be unresponsive - fine, cleanup below handles it

def _kill_process_tree():
    """Kill the ollama serve process group AND any orphaned runner processes."""
    global server
    if server is not None:
        try:
            os.killpg(os.getpgid(server.pid), signal.SIGKILL)
            server.wait(timeout=5)
        except Exception:
            pass
    # Safety net - catches orphaned model-runner processes by name
    subprocess.run(["pkill", "-9", "-f", "ollama runner"], stderr=subprocess.DEVNULL)
    subprocess.run(["pkill", "-9", "-f", "ollama_llama_server"], stderr=subprocess.DEVNULL)
    subprocess.run(["pkill", "-9", "-f", "ollama serve"], stderr=subprocess.DEVNULL)
    time.sleep(1)

def start_ollama():
    global server
    server = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True   # own process group -> no orphaned children later
    )
    time.sleep(5)
    print("✅ Ollama Server Started")

def restart_ollama():
    print("\n🔄 Restarting Ollama Server...")
    _unload_model()
    _kill_process_tree()
    time.sleep(2)
    start_ollama()
    print("✅ Ollama Restarted Successfully")

start_ollama()

In [ ]:
!ollama pull {MODEL}
print(f"✅ {MODEL} Ready")

In [ ]:
!ollama list

In [ ]:
# RAM sanity check baseline - note this number, compare after restarts later
!free -h
!ps aux | grep -i ollama

In [ ]:
df = pd.read_excel(DATASET)
print(df.head())
print(f"\nTotal Questions : {len(df)}")

In [ ]:
def ask_model(question, timeout=120):
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        options={
            "temperature": 0.4,
            "seed": 42,
            "num_predict": 1000
        }
    )
    return response.message.content.strip()

In [ ]:
# ----------------------------------------------------------
# Checkpoint filename
# ----------------------------------------------------------
safe_model = re.sub(r'[^A-Za-z0-9_-]', '_', MODEL)
checkpoint_dir = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)

CHECKPOINT = os.path.join(checkpoint_dir, f"{safe_model}_checkpoint.xlsx")

results = []
start_index = 0

if os.path.exists(CHECKPOINT):
    old = pd.read_excel(CHECKPOINT)
    results = old.to_dict("records")
    start_index = len(results)
    print(f"✅ Resuming from Question {start_index}")
else:
    print("✅ Starting New Benchmark")

# ----------------------------------------------------------
# Evaluation loop
# ----------------------------------------------------------
for i in tqdm(range(start_index, len(df)), desc="Benchmarking"):

    row = df.iloc[i]

    try:
        prediction = ask_model(row["Question"])
    except Exception as e:
        prediction = f"ERROR: {str(e)}"

    results.append({
        "ID": row["S No"],
        "Question": row["Question"],
        "Ground Truth": row["Answer"],
        "Prediction": prediction,
        "Model": MODEL
    })

    if (i + 1) % CHECKPOINT_INTERVAL == 0:
        pd.DataFrame(results).to_excel(CHECKPOINT, index=False)
        print(f"💾 Checkpoint Saved ({i + 1}/{len(df)})")

    if (i + 1) % RESTART_INTERVAL == 0:
        restart_ollama()
        # verify RAM didn't accumulate - print after each restart
        !free -h

# ----------------------------------------------------------
# Final save
# ----------------------------------------------------------
pd.DataFrame(results).to_excel(CHECKPOINT, index=False)
print("\n💾 Final Checkpoint Saved")
print("✅ Evaluation Completed")

In [ ]:
dataset_name = os.path.splitext(os.path.basename(DATASET))[0]
model_name = re.sub(r'[^A-Za-z0-9_-]', '_', MODEL)

results_dir = os.path.join(OUTPUT_DIR, "results")
os.makedirs(results_dir, exist_ok=True)

output_filename = os.path.join(results_dir, f"{dataset_name}_{model_name}_results.xlsx")

output = pd.DataFrame(results)
output.to_excel(output_filename, index=False)

print("\n✅ Benchmark Finished!")
print(f"Saved as : {output_filename}")

In [ ]:
!ollama ps
!ollama list

In [ ]:
_kill_process_tree()
print("✅ Fully cleaned up")

In [ ]:
import subprocess

result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
lines = result.stdout.strip().split("\n")[1:]  # skip header row

for line in lines:
    model_name = line.split()[0]
    subprocess.run(["ollama", "rm", model_name])
    print(f"🗑️ Removed: {model_name}")

print("✅ All models removed")